# Run MOSART for a QGIS landslide job

Reconstructs **elevation change (Δh)** from Sentinel-1 amplitude with the public
[MOSART](https://github.com/uafgeotools/MOSART) package (MIT), and writes a
georeferenced **Δh GeoTIFF** back into your project's `mosart/` folder on Google
Drive — where the QGIS Volume tab auto-imports it for the ∫Δh fit.

**This notebook is import-only:** it *uses* `mosart` as a library. It does not
modify the package or anyone else's code.

**You need:** a free [NASA Earthdata Login](https://urs.earthdata.nasa.gov/) (for
the Sentinel-1 SLC download). The Copernicus GLO-30 DEM is fetched automatically.

**Runtime:** use a Linux GPU/High-RAM runtime if you can — ISCE2 coregistration is
memory-hungry. Expect the coregistration cell to take a while.

> ⚠️ **Not turnkey.** Two cells need your judgement: confirming the **bursts**
> `asf_search` finds, and the **weight/water mask**. They're marked **TUNE**.

## 1 · Install (ISCE2 needs conda; this restarts the runtime once)

In [ ]:
# condacolab restarts the kernel — that's expected. Re-run from the NEXT cell after.
!pip -q install condacolab
import condacolab
condacolab.install()

In [ ]:
# After the restart, run this to install ISCE2 + MOSART + friends.
!mamba install -q -y -c conda-forge isce2 gdal h5py scikit-image
!pip -q install mosart hyp3-isce2 asf_search rasterio
print('installed')

## 2 · Mount Drive and load the QGIS job

Set `MOSART_DIR` to your project's `mosart/` folder on Drive (the one the QGIS
button wrote the job into).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, json, os

# EDIT THIS to your project's mosart/ folder on Drive:
MOSART_DIR = '/content/drive/Shareddrives/aec_research/Ethan_Hasenauer/QGIS/2026-08-08 Illiamna/mosart'

jobs = sorted(glob.glob(os.path.join(MOSART_DIR, '*_job.json')), key=os.path.getmtime)
assert jobs, f'no *_job.json in {MOSART_DIR} — press "Prepare MOSART job" in QGIS first'
job = json.load(open(jobs[-1]))
print('job:', json.dumps(job, indent=2))
lon0, lat0, lon1, lat1 = job['aoi_lonlat_bbox']
os.makedirs('data', exist_ok=True)

## 3 · Find Sentinel-1 bursts (TUNE — confirm the result)

Searches ASF for IW bursts over the AOI in the pre/post window. MOSART needs
**one burst id across several dates**. Inspect the table, pick the burst id with
the best temporal coverage spanning the event, and keep those in `chosen`.

In [ ]:
import asf_search as asf
from datetime import datetime, timedelta

ev = datetime.fromisoformat(job['event_date'])
start = (ev - timedelta(days=job['pre_window_days'])).strftime('%Y-%m-%d')
end   = (ev + timedelta(days=job['post_window_days'])).strftime('%Y-%m-%d')
flightdir = {'ascending': asf.FLIGHT_DIRECTION.ASCENDING,
             'descending': asf.FLIGHT_DIRECTION.DESCENDING}.get(job['orbit_direction'])

opts = dict(dataset=asf.DATASET.SLC_BURST, intersectsWith=
            f'POINT({(lon0+lon1)/2} {(lat0+lat1)/2})',
            start=start, end=end, polarization=job.get('polarization','VV'))
if flightdir: opts['flightDirection'] = flightdir
results = asf.search(**opts)

rows = []
for r in results:
    p = r.properties
    rows.append((p.get('burst',{}).get('fullBurstID'), p['startTime'][:10],
                 p['flightDirection'], p['pathNumber'], p['fileID']))
import collections
byburst = collections.defaultdict(list)
for bid, date, fd, path, fid in rows:
    byburst[bid].append((date, fid))
for bid, items in sorted(byburst.items(), key=lambda kv: -len(kv[1])):
    print(f'{bid}: {len(items)} dates  {[d for d,_ in sorted(items)]}')

In [ ]:
# TUNE: set BURST_ID to the id above with the most dates spanning the event.
BURST_ID = None   # e.g. '202224_IW2'  <-- fill this in from the list above
assert BURST_ID, 'pick a burst id from the printout above'
chosen = sorted(byburst[BURST_ID])
print(f'{len(chosen)} scenes for {BURST_ID}:', [d for d,_ in chosen])
assert len(chosen) >= 3, 'MOSART wants several dates; widen the window if too few'

## 4 · Earthdata credentials + write the burst list

`coregistration_bursts` downloads the bursts itself (via hyp3-isce2), so we give
it credentials through `~/.netrc` and hand it the burst ids.

**Type it once, securely:** add `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD` in
the Colab **Secrets** panel (the 🔑 icon, left sidebar) and toggle notebook
access — this cell reads them and never asks again. They live in your Google
account's secret store, NOT in the notebook, Drive, or QGIS. No secrets set?
It falls back to a one-time prompt.

Get a free login at https://urs.earthdata.nasa.gov/ .

In [ ]:
import os, getpass
user = pw = None
try:
    from google.colab import userdata
    user = userdata.get('EARTHDATA_USERNAME')
    pw = userdata.get('EARTHDATA_PASSWORD')
    print('using Earthdata creds from Colab Secrets')
except Exception:
    pass
if not user or not pw:
    print('No Colab Secrets found — add EARTHDATA_USERNAME/PASSWORD via the 🔑',
          'panel to skip this next time. Entering manually for now:')
    user = user or input('Earthdata username: ')
    pw = pw or getpass.getpass('Earthdata password: ')
netrc = os.path.expanduser('~/.netrc')
with open(netrc, 'w') as fh:
    fh.write(f'machine urs.earthdata.nasa.gov login {user} password {pw}\n')
os.chmod(netrc, 0o600)

with open('bursts_list_sen1.txt', 'w') as fh:
    fh.write('\n'.join(fid for _, fid in chosen) + '\n')
print('wrote ~/.netrc and bursts_list_sen1.txt:', [fid for _, fid in chosen])

## 5 · Coregister (SLOW — ISCE2)

`mosart.sfs.coregistration_bursts` coregisters the stack and geocodes GLO-30 into
radar coordinates. This is the heavy step.

In [ ]:
from mosart import sfs, lsquares
import numpy as np, h5py

cor_stack, pre_stack = 'projections.h5', 'preprocessed.h5'
sfs.coregistration_bursts('bursts_list_sen1.txt', output=cor_stack)
print('coregistered ->', cor_stack)

## 6 · AOI in radar coords, then preprocess

In [ ]:
h5i = h5py.File(cor_stack, 'r')
lonrdr, latrdr = h5i['lon'][:], h5i['lat'][:]
h5i.close()
x0, y0, xs, ys = sfs.get_box(lonrdr, latrdr, lons=[lon0, lon1], lats=[lat0, lat1])
XS, YS = [x0, x0 + xs], [y0, y0 + ys]
print('radar AOI  range', XS, ' azimuth', YS)

sigma_amp = 20   # TUNE for Sentinel-1 (author default 20)
patch_kw = dict(patch_size=5, patch_distance=5)
sfs.preprocessing(xs=XS, ys=YS, projections=cor_stack, output=pre_stack,
                  sigma_amp=sigma_amp, patch_kw=patch_kw)
print('preprocessed ->', pre_stack)

## 7 · Weight matrix (TUNE)

A landslide on rock/ice has no standing water, so a plain weight of 1 is a safe
start (unlike the volcano example's water masking). Refine if amplitude artefacts
bias the fit.

In [ ]:
h5i = h5py.File(pre_stack, 'r')
keys = [k for k in h5i.keys() if k.isdigit()]
tamps = np.array([h5i[k][:] for k in keys])
h5i.close()
weights = np.ones(tamps.shape)     # TUNE: per-date/-pixel weights if needed
waters = np.full(tamps.shape, False)
print(len(keys), 'dates:', keys)

## 8 · Reconstruct per-date DEMs and difference

In [ ]:
import multiprocessing as mp
from functools import partial

def onedem(i):
    demdef, _grd, _w = lsquares.getdem(keys[i], h5file=pre_stack,
                                       weight=weights[i], water=waters[i])
    return demdef
with mp.Pool(processes=max(1, os.cpu_count())) as pool:
    demdefs = pool.map(onedem, range(len(keys)))
demdefs = np.array(demdefs)

# reference = the earliest PRE-event date; post = the last date after the event
dates = [datetime.fromisoformat(f'{k[:4]}-{k[4:6]}-{k[6:8]}') for k in keys]
ref = min(range(len(keys)), key=lambda i: dates[i])
post = max(range(len(keys)), key=lambda i: dates[i])
change = demdefs[post] - demdefs[ref]   # metres, radar coords
print(f'Δh: {keys[ref]} -> {keys[post]}  range [{np.nanmin(change):.1f}, {np.nanmax(change):.1f}] m')

## 9 · Georeference and save the Δh GeoTIFF back to Drive

Writes into `mosart/` so the QGIS Volume tab auto-imports it. If it lands
upside-down in QGIS, flip `north_up` below.

In [ ]:
import rasterio
from rasterio.transform import from_bounds

pixel_size = 0.0005   # deg (~50 m); shrink for finer output
geochange, extent = sfs.georeference(pre_stack, array=change, pixel_size=pixel_size)
minlon, maxlon, minlat, maxlat = extent

north_up = True
arr = geochange if north_up else geochange[::-1]
h, w = arr.shape
transform = from_bounds(minlon, minlat, maxlon, maxlat, w, h)

out = os.path.join(MOSART_DIR, job['output_dh_geotiff'])
with rasterio.open(out, 'w', driver='GTiff', height=h, width=w, count=1,
                   dtype='float32', crs='EPSG:4326', transform=transform,
                   nodata=-9999, compress='deflate') as d:
    d.write(np.where(np.isfinite(arr), arr, -9999).astype('float32'), 1)
print('WROTE', out)
print('Back in QGIS: press ↻ Refresh layers (or it auto-imports), pick it in the',
      'Elevation change (Δh) box, choose the ∫Δh fit, and Measure.')